# Milestone 3 - Hybrid vector store (BGE-M3 + Qdrant)

**BGE-M3** (XLM-RoBERTa-large encoder) produces, in a single forward pass, a 1024-d dense vector (normalised [CLS]) and a sparse lexical vector (ReLU of a learned per-token weight, max-pooled per token id). Both are stored in one **Qdrant** collection with an HNSW index and **int8 scalar quantisation**.

In [1]:
import os, sys, json, time, warnings, pathlib
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
from rag import config

In [2]:
from rag import parsing, store
from rag.embedder import get_embedder
chunks = parsing.load_chunks()
emb = get_embedder()
t = time.time()
dense, sparse = emb.encode([c.text for c in chunks])
print(f"encoded {len(chunks)} chunks in {time.time() - t:.1f}s on {emb.device}; dense shape {dense.shape}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

encoded 273 chunks in 2.4s on mps; dense shape (273, 1024)


### What the sparse head learns
Top-weighted tokens for one chunk - the model up-weights rare, discriminative terms (model codes, numbers).

In [3]:
i = next(k for k, c in enumerate(chunks) if c.kind == "table" and c.doc_type == "docx")
top = sorted(sparse[i].items(), key=lambda kv: -kv[1])[:12]
print(chunks[i].text[:200], "...\n")
pd.DataFrame([(emb.tok.convert_ids_to_tokens(t), round(w, 3)) for t, w in top], columns=["token", "weight"])

[HX Centrifugal Pump Series - Technical Specification | 2. HX-110 Centrifugal Pump]
| Parameter | Value |
|---|---|
| Maximum flow rate | 1865 L/min |
| Maximum head | 74 m |
| Motor power | 14.1 kW | ...



,token,weight
0,110,0.277
1,▁Pump,0.247
2,X,0.241
3,fu,0.234
4,▁1865,0.227
5,▁head,0.206
6,▁flow,0.205
7,eller,0.197
8,▁Centri,0.187
9,▁fluid,0.183


In [4]:
import numpy as np
print("mean non-zero sparse terms per chunk:", round(np.mean([len(s) for s in sparse]), 1))
print("dense L2 norms (should be 1):", np.linalg.norm(dense, axis=1)[:5].round(4))

mean non-zero sparse terms per chunk: 48.0
dense L2 norms (should be 1): [1. 1. 1. 1. 1.]


### Create the collection and index

In [5]:
print(store.ensure_qdrant())
qc = store.client()
store.create_collection(qc)
n = store.index_chunks(qc, chunks, dense, sparse)
info = qc.get_collection(config.COLLECTION)
print("indexed points:", n, "| status:", info.status)
cfg = info.config
print("dense:", cfg.params.vectors["dense"].size, cfg.params.vectors["dense"].distance, "| HNSW m =", cfg.hnsw_config.m, "ef_construct =", cfg.hnsw_config.ef_construct)
print("quantisation:", cfg.quantization_config)
print("sparse vectors:", list(cfg.params.sparse_vectors.keys()))

already running


indexed points: 273 | status: green
dense: 1024 Cosine | HNSW m = 16 ef_construct = 100
quantisation: scalar=ScalarQuantizationConfig(type=<ScalarType.INT8: 'int8'>, quantile=0.99, always_ram=True, memory=None)
sparse vectors: ['sparse']


### Memory effect of int8 quantisation (dense vectors)

In [6]:
f32 = len(chunks) * config.DENSE_DIM * 4
i8 = len(chunks) * config.DENSE_DIM * 1
print(f"float32: {f32/1024:.0f} KiB   int8: {i8/1024:.0f} KiB   (4x smaller; originals kept on disk for rescoring)")
for n_docs in (1e5, 1e6, 1e7):
    print(f"  at {int(n_docs):>10,} chunks: float32 {n_docs*4096/2**30:6.1f} GiB  ->  int8 {n_docs*1024/2**30:5.1f} GiB")

float32: 1092 KiB   int8: 273 KiB   (4x smaller; originals kept on disk for rescoring)
  at    100,000 chunks: float32    0.4 GiB  ->  int8   0.1 GiB
  at  1,000,000 chunks: float32    3.8 GiB  ->  int8   1.0 GiB
  at 10,000,000 chunks: float32   38.1 GiB  ->  int8   9.5 GiB


### Quantised search agrees with exact search

In [7]:
from qdrant_client import models
q = "maximum head of the HX-320 pump"
qd, qs = emb.encode([q])
approx = [p.payload["chunk_id"] for p in store.search_dense(qc, qd[0], 5)]
exact = [p.payload["chunk_id"] for p in qc.query_points(config.COLLECTION, query=qd[0].tolist(), using="dense", limit=5,
         search_params=models.SearchParams(exact=True, quantization=models.QuantizationSearchParams(ignore=True))).points]
print("HNSW+int8:", approx); print("exact    :", exact); print("overlap@5:", len(set(approx) & set(exact)) / 5)

HNSW+int8: ['HX_Pump_Selection_Guide.md::7::0', 'HX_Pump_Series_Technical_Specification.docx::8::1', 'HX_Pump_Series_Technical_Specification.docx::12::1', 'HX_Pump_Selection_Guide.md::11::0', 'HX_Pump_Series_Technical_Specification.docx::3::1']
exact    : ['HX_Pump_Selection_Guide.md::7::0', 'HX_Pump_Series_Technical_Specification.docx::8::1', 'HX_Pump_Series_Technical_Specification.docx::12::1', 'HX_Pump_Selection_Guide.md::11::0', 'HX_Pump_Series_Technical_Specification.docx::3::1']
overlap@5: 1.0
